# Exercise 4: Filtering synthetic data

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Cross-Lingual Transfer · version 0.1.0*

> Compare unfiltered, quality-filtered and randomly reduced synthetic parallel data at the same retained size, scoring each retained set against real target references and checking what the filter selects for.

**Setup.** `pip install sklearn-crfsuite scikit-learn sacrebleu torch transformers sentencepiece fasttext-wheel "stanza==1.11.0" requests "turkicnlp[hfst]==0.5.0"`; each notebook uses only part of this. Data come from commit-pinned GitHub repositories and are hash-checked: KazNERD and KyrgyzNER (Exercise 1), SIB-200 (Exercises 2 and 3), the X-WMT Kazakh–Kyrgyz test set (Exercise 4) and the UD 2.17 Turkish–German SAGT and Turkish IMST test files (Exercise 5). Sentence encoders and NLLB-200 are read from local copies (`MODELS_DIR`, `HF_HUB_DIR`, `NLLB_PATH`) or the Hub at the revisions pinned in the code; GlotLID from `GLOTLID_PATH`. Run the notebooks from the `exercises/` folder: the first code cell imports the chapter's helpers from `../code/remaining/transfer.py`. Embeddings, translations and model outputs are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`). Everything runs on the CPU.

NLLB-200 (600M, greedy) translates the 500 Kazakh sentences of the X-WMT Kazakh–Kyrgyz test set into Kyrgyz and back. The filter keeps the half with the highest round-trip chrF. Because the real Kyrgyz references exist, each retained half can be graded directly; random and length-matched random halves of the same size are the baselines.

In [ ]:
import collections, csv, hashlib, io, json, math, os, random, re, statistics, sys
from pathlib import Path
import numpy as np, requests
sys.path.insert(0, str(Path("../code/remaining").resolve()))
from transfer import SCHEMA, bio_spans, conversion_map, restore_span

CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
THREADS = int(os.environ.get("THREADS", os.cpu_count() or 1))
KAZNERD = "https://raw.githubusercontent.com/IS2AI/KazNERD/bd4333d0f5952b9fafb2ef2ac2fefa0ad3c0333f/KazNERD/IOB2_{}.txt"
KYRGYZNER = "https://raw.githubusercontent.com/Akyl-AI/KyrgyzNER/b76b9c9c9c4f5d681c25308cc0d8dd9f0bd6569c/data/{}"
SIB = "https://raw.githubusercontent.com/dadelani/sib-200/efad32e0fc601cfbfdf5ca80672f29697f6f401f/data/annotated/"
TIL = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/7d988cc47dba889d02262d893bf9a46e43a6bb36/xwmt/test/"
UD = "https://raw.githubusercontent.com/UniversalDependencies/{}/r2.17/{}"
SHA256 = {
    KAZNERD.format("train"): "e5a80f6d4bbff499a4bf7cbf96a169feb761411151fb45abebeaae06995ff647",
    KYRGYZNER.format("KyrgyzNER_TEST.jsonl.gold.txt"): "57a35a92139497819a2792d3e9e00a41b387b029b7851be7a64f4968befaddf5",
    UD.format("UD_Turkish_German-SAGT", "qtd_sagt-ud-test.conllu"): "0bbb06de66a490bfbb93d103a03d0d89e3835baf0af755a57b5fa5f885ae01bc",
    UD.format("UD_Turkish-IMST", "tr_imst-ud-test.conllu"): "fc9269bd598489b942a025b678779557266ce0c054a843e47c7db386d76d6459",
}
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    h = hashlib.sha256(path.read_bytes()).hexdigest(); FETCHED[url] = h
    if url in SHA256 and h != SHA256[url]:
        raise ValueError(f"{url} differs from the pinned file")
    return path

_CTS = {}
def to_cts(lang, text):
    """Common Turkic Script view (TurkicNLP transliterator); Turkish and English stay unchanged."""
    if lang in ("tur", "eng"):
        return text
    if lang not in _CTS:
        from turkicnlp.scripts import Script
        from turkicnlp.scripts.transliterator import Transliterator
        src = Script.LATIN if lang in ("uzb", "aze") else Script.CYRILLIC
        _CTS[lang] = Transliterator(lang, src, Script.COMMON_TURKIC)
    return _CTS[lang].transliterate(text)

def sib(lang):
    """All 1,004 SIB-200 rows of a language as (index_id, split, topic, text), sorted by index_id."""
    rows = []
    for split in ("train", "dev", "test"):
        text = fetch(f"{SIB}{lang}/{split}.tsv").read_text(encoding="utf-8")
        rows += [(int(r["index_id"]), split, r["category"], r["text"]) for r in csv.DictReader(io.StringIO(text), delimiter="\t")]
    return sorted(rows)

MODELS = {"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": "e8f8c211226b894fcb81acc59f3b34ba3efd5f42",
          "sentence-transformers/paraphrase-xlm-r-multilingual-v1": "c2e19453aae46f98a865abfdf173f35f2e119f5a"}
NAMES = {"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": "MiniLM",
         "sentence-transformers/paraphrase-xlm-r-multilingual-v1": "XLM-R para", "nllb": "NLLB enc"}
NLLB_PATH = os.environ.get("NLLB_PATH", str(Path("~/.turkicnlp/models/huggingface/facebook--nllb-200-distilled-600M").expanduser()))
NLLB_CODE = {"kaz_Cyrl": "kaz_Cyrl", "kir_Cyrl": "kir_Cyrl", "tat_Cyrl": "tat_Cyrl", "bak_Cyrl": "bak_Cyrl", "uzn_Latn": "uzn_Latn",
             "tur_Latn": "tur_Latn", "rus_Cyrl": "rus_Cyrl", "eng_Latn": "eng_Latn", "fra_Latn": "fra_Latn", "azj_Latn": "azj_Latn"}
def model_source(name):
    rev = MODELS[name]
    for p in (Path(os.environ.get("HF_HUB_DIR", "~/.cache/huggingface/hub")).expanduser() / f"models--{name.replace('/', '--')}" / "snapshots" / rev,
              Path(os.environ.get("MODELS_DIR", CACHE / "models")) / f"models--{name.replace('/', '--')}--{rev}"):
        if (p / "config.json").exists():
            return str(p), None
    return name, rev
_ENC = {}
def embed(model, texts, lang=None, batch=32):
    """L2-normalised mean-pooled sentence vectors, as in Meaning and Retrieval (same cache)."""
    key = hashlib.sha1(json.dumps([model, lang, texts], ensure_ascii=False).encode()).hexdigest()[:20]
    path = CACHE / "ch13-vectors" / f"{key}.npy"
    if path.exists():
        return np.load(path)
    import torch
    torch.set_num_threads(THREADS)
    if model not in _ENC:
        from transformers import AutoModel, AutoTokenizer
        if model == "nllb":
            from transformers import AutoModelForSeq2SeqLM
            _ENC[model] = (AutoTokenizer.from_pretrained(NLLB_PATH), AutoModelForSeq2SeqLM.from_pretrained(NLLB_PATH).get_encoder().eval(), 512)
        else:
            src, rev = model_source(model)
            _ENC[model] = (AutoTokenizer.from_pretrained(src, revision=rev), AutoModel.from_pretrained(src, revision=rev).eval(), 128)
    tok, enc, limit = _ENC[model]
    if model == "nllb":
        tok.src_lang = NLLB_CODE[lang]
    out = []
    with torch.inference_mode():
        for i in range(0, len(texts), batch):
            b = tok(texts[i:i + batch], padding=True, truncation=True, max_length=limit, return_tensors="pt")
            h = enc(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1).to(h.dtype)
            out.append(torch.nn.functional.normalize((h * m).sum(1) / m.sum(1).clamp(min=1), dim=-1).numpy())
    vec = np.concatenate(out).astype(np.float32)
    path.parent.mkdir(parents=True, exist_ok=True); np.save(path, vec)
    return vec

def macro_f1(gold, pred, labels):
    from sklearn.metrics import f1_score
    return f1_score(gold, pred, labels=labels, average="macro", zero_division=0)

print("helpers loaded from code/remaining/transfer.py; schema map:", SCHEMA)

In [ ]:
import sacrebleu, torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
def detok(s):
    s = re.sub(r" ([’'] ?)", lambda m: m.group(1).strip(), s); s = re.sub(r" ([.,!?;:%)\]»])", r"\1", s)
    return re.sub(r"([(\[«]) ", r"\1", s).strip()
kk = [detok(s) for s in fetch(TIL + "kk-ky/kk-ky.kk").read_text(encoding="utf-8").splitlines()]
ky = [detok(s) for s in fetch(TIL + "kk-ky/kk-ky.ky").read_text(encoding="utf-8").splitlines()]
TRANS = CACHE / "ch16-nllb.json"
memo = json.loads(TRANS.read_text()) if TRANS.exists() else {}
def translate(sents, src, tgt):
    todo = [s for s in sents if f"{src}|{tgt}|{s}" not in memo]
    if todo:
        torch.set_num_threads(THREADS)
        tok = AutoTokenizer.from_pretrained(NLLB_PATH, src_lang=src); m = AutoModelForSeq2SeqLM.from_pretrained(NLLB_PATH).eval()
        with torch.inference_mode():
            for i in range(0, len(todo), 8):
                b = tok(todo[i:i + 8], return_tensors="pt", padding=True)
                g = m.generate(**b, forced_bos_token_id=tok.convert_tokens_to_ids(tgt), num_beams=1, max_new_tokens=200, max_length=None)
                for s, t in zip(todo[i:i + 8], tok.batch_decode(g, skip_special_tokens=True)):
                    memo[f"{src}|{tgt}|{s}"] = t
                TRANS.write_text(json.dumps(memo, ensure_ascii=False))          # saved after every batch, so a run can resume
    return [memo[f"{src}|{tgt}|{s}"] for s in sents]
# Synthetic Kyrgyz for each Kazakh sentence (forward), and its back-translation for a round-trip score.
syn = translate(kk, "kaz_Cyrl", "kir_Cyrl")
back = translate(syn, "kir_Cyrl", "kaz_Cyrl")
rt = [sacrebleu.sentence_chrf(b, [k]).score for b, k in zip(back, kk)]
true = [sacrebleu.sentence_chrf(s, [r]).score for s, r in zip(syn, ky)]       # quality against the real Kyrgyz reference
copy = [sacrebleu.sentence_chrf(s, [k]).score for s, k in zip(syn, kk)]       # similarity to the Kazakh source
print(f"X-WMT kk-ky: {len(kk)} sentence pairs; NLLB-200 600M greedy Kazakh->Kyrgyz and back")
print(f"synthetic Kyrgyz identical to the Kazakh source: {sum(s == k for s, k in zip(syn, kk))}")
keep = len(kk) // 2
order = sorted(range(len(kk)), key=lambda i: -rt[i])
rng = random.Random(0)
sets = {"all": list(range(len(kk))), "top half by round-trip chrF": order[:keep]}
rand = [rng.sample(range(len(kk)), keep) for _ in range(20)]
from scipy.stats import spearmanr
print(f"round-trip chrF vs chrF against the real Kyrgyz reference: Spearman {spearmanr(rt, true).statistic:.2f}")
print(f"{'subset':<30}{'n':>5}{'chrF vs ref':>13}{'copy chrF':>11}{'words':>7}")
for name, ids in sets.items():
    print(f"{name:<30}{len(ids):>5}{statistics.mean(true[i] for i in ids):13.1f}{statistics.mean(copy[i] for i in ids):11.1f}"
          f"{statistics.mean(len(kk[i].split()) for i in ids):7.1f}")
r = [statistics.mean(true[i] for i in ids) for ids in rand]
print(f"{'random half (20 draws)':<30}{keep:>5}{statistics.mean(r):13.1f}  (range {min(r):.1f}-{max(r):.1f})"
      f"{statistics.mean(statistics.mean(copy[i] for i in ids) for ids in rand):11.1f}{statistics.mean(statistics.mean(len(kk[i].split()) for i in ids) for ids in rand):7.1f}")
# A length-matched random baseline: the filter prefers short sentences.
bins = collections.defaultdict(list)
for i in range(len(kk)): bins[min(len(kk[i].split()) // 5, 6)].append(i)
need = collections.Counter(min(len(kk[i].split()) // 5, 6) for i in order[:keep])
matched = [statistics.mean(true[i] for b, n in need.items() for i in random.Random(s).sample(bins[b], n)) for s in range(20)]
print(f"{'length-matched random half':<30}{keep:>5}{statistics.mean(matched):13.1f}")

## Worked solution

Output of `experiments/ch16_exercises/run_ch16_exercises.py`:

```
X-WMT kk-ky: 500 sentence pairs; NLLB-200 600M greedy Kazakh->Kyrgyz and back
synthetic Kyrgyz identical to the Kazakh source: 0
round-trip chrF vs chrF against the real Kyrgyz reference: Spearman 0.31
subset                            n  chrF vs ref  copy chrF  words
all                             500         44.0       32.2   16.9
top half by round-trip chrF     250         47.2       36.1   17.5
random half (20 draws)          250         44.0  (range 42.9-45.3)       32.1   16.9
length-matched random half      250         44.3
```

**Filtering selects a subset with higher reference overlap.** Mean sentence chrF against Kyrgyz references is 44.0 for all 500 outputs and 47.2 for the top 250 by round-trip chrF. Twenty random halves range from 42.9 to 45.3; the length-bin-matched baseline averages 44.3. The random range is not a confidence interval for the filter's effect.

The selected outputs also have higher overlap with the Kazakh source (36.1 versus 32.2). None is an exact copy. Shared vocabulary, translation choices and copying could all affect overlap, so this does not establish that the filter selects erroneous copies. Round-trip and reference chrF correlate at Spearman 0.31; reference chrF is itself an overlap metric, not a human quality judgement.

No model is adapted here. The experiment selects among benchmark test outputs, so these pairs must remain outside training. To test whether filtering improves an adapted model, use separate synthetic training data, choose the filter on development data and evaluate on untouched references.

## Your turn — extension

Add a language-identification filter (GlotLID, Kyrgyz vs Kazakh) and a length-ratio filter, compare each with random halves of the same size, then repeat the filtering on separate synthetic training data. Adapt NLLB on those training subsets and evaluate on an untouched test set; do not train on these X-WMT test pairs.